# Part 21: CrewAI — Multi-Agent Teams

> Build collaborative agent teams with defined roles, goals, and backstories. CrewAI enables complex workflows through role-based agents working toward shared objectives.

---


## 21.1 CrewAI Core Concepts

| Concept | Description |
|---------|-------------|
| **Agent** | An AI worker with a role, goal, and backstory |
| **Task** | A specific piece of work assigned to an agent |
| **Crew** | A team of agents working together |
| **Process** | How tasks are executed (sequential / hierarchical) |
| **Tool** | External capability an agent can use |


In [ ]:
# pip install crewai crewai-tools
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from langchain_openai import ChatOpenAI
import os


## 21.2 Defining Agents

In [ ]:
from crewai import Agent

# Each agent has: role, goal, backstory, tools, llm
researcher = Agent(
    role="Senior Research Analyst",
    goal="Uncover cutting-edge developments in AI and data science",
    backstory="""You work at a leading tech think tank.
Your expertise lies in identifying emerging trends.
You have a knack for dissecting complex data and presenting actionable insights.""",
    verbose=True,
    allow_delegation=False,
    tools=[],  # add tools here
    llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7)
)

writer = Agent(
    role="Tech Content Strategist",
    goal="Craft compelling content on tech advancements",
    backstory="""You are a renowned Content Strategist, known for your insightful
and engaging articles. You transform complex concepts into compelling narratives.""",
    verbose=True,
    allow_delegation=True
)

editor = Agent(
    role="Senior Editor",
    goal="Edit and polish content to the highest standards",
    backstory="""You are an experienced editor with a sharp eye for detail.
You ensure all content is accurate, engaging, and publication-ready.""",
    verbose=True,
    allow_delegation=False
)


## 21.3 Defining Tasks

In [ ]:
from crewai import Task

# Tasks: description, expected_output, agent, context (dependencies)
research_task = Task(
    description="""Conduct a comprehensive analysis of the latest AI advancements in 2024.
    Identify key trends, breakthrough technologies, and potential industry impacts.
    Focus on: LLMs, multimodal AI, AI agents, and AI safety.""",
    expected_output="""A detailed research report with:
    - Top 5 AI trends of 2024
    - Key players and their contributions  
    - Potential industry disruptions
    - Future predictions""",
    agent=researcher
)

writing_task = Task(
    description="""Using the research report, develop a compelling blog post 
    about the most significant AI advancements of 2024.
    Make it engaging, accessible, and informative for a tech-savvy audience.""",
    expected_output="A 1500-word blog post with an engaging title, introduction, body sections, and conclusion.",
    agent=writer,
    context=[research_task]  # depends on research being done first
)

editing_task = Task(
    description="""Review and edit the blog post for clarity, accuracy, and engagement.
    Ensure it meets publication standards. Check for grammar, flow, and factual accuracy.""",
    expected_output="A polished, publication-ready blog post.",
    agent=editor,
    context=[writing_task]
)


## 21.4 Assembling the Crew

In [ ]:
from crewai import Crew, Process

# Sequential: tasks run one after another
crew_sequential = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,
    verbose=True
)

# Run the crew
# result = crew_sequential.kickoff()
# print(result)

# Hierarchical: a manager LLM delegates to agents
crew_hierarchical = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.hierarchical,
    manager_llm=ChatOpenAI(model_name="gpt-4o", temperature=0.2),
    verbose=True
)


## 21.5 CrewAI Tools

In [ ]:
from crewai.tools import BaseTool
from pydantic import Field
import requests
from bs4 import BeautifulSoup

class WebScraperTool(BaseTool):
    name: str = "Web Scraper"
    description: str = "Scrape content from a webpage URL"
    
    def _run(self, url: str) -> str:
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, "html.parser")
            for tag in soup(["script", "style", "nav", "footer"]):
                tag.decompose()
            return soup.get_text(separator="\n", strip=True)[:3000]
        except Exception as e:
            return f"Error scraping {url}: {e}"

class CalculatorTool(BaseTool):
    name: str = "Calculator"
    description: str = "Perform mathematical calculations"
    
    def _run(self, expression: str) -> str:
        try:
            import ast
            # Safe evaluation
            tree = ast.parse(expression, mode='eval')
            result = eval(compile(tree, '<string>', 'eval'))
            return str(result)
        except Exception as e:
            return f"Calculation error: {e}"

# Attach tools to agents
research_agent_with_tools = Agent(
    role="Research Analyst",
    goal="Research and analyze topics thoroughly",
    backstory="Expert researcher with web access",
    tools=[WebScraperTool(), CalculatorTool()],
    verbose=True
)


## 21.6 Engineering Team Example

In [ ]:
# A software engineering crew: PM, Developer, QA

product_manager = Agent(
    role="Product Manager",
    goal="Define clear requirements and acceptance criteria",
    backstory="Experienced PM who bridges business needs and technical implementation.",
    verbose=True
)

developer = Agent(
    role="Senior Software Engineer",
    goal="Write clean, efficient, well-tested code",
    backstory="Full-stack engineer with 10 years experience in Python and cloud systems.",
    verbose=True
)

qa_engineer = Agent(
    role="QA Engineer",
    goal="Ensure software quality through comprehensive testing",
    backstory="Quality expert who thinks like an adversarial user.",
    verbose=True
)

# Tasks
requirements_task = Task(
    description="Define requirements for a REST API that estimates product prices.",
    expected_output="Detailed technical requirements with API spec and acceptance criteria.",
    agent=product_manager
)

implementation_task = Task(
    description="Implement the price estimation API based on the requirements.",
    expected_output="Working Python code with FastAPI, including endpoint definitions and business logic.",
    agent=developer,
    context=[requirements_task]
)

testing_task = Task(
    description="Write comprehensive tests for the implemented API.",
    expected_output="Test suite with unit tests, integration tests, and edge cases.",
    agent=qa_engineer,
    context=[implementation_task]
)

engineering_crew = Crew(
    agents=[product_manager, developer, qa_engineer],
    tasks=[requirements_task, implementation_task, testing_task],
    process=Process.sequential,
    verbose=True
)


## 21.7 Dynamic Crew with Gradio UI

In [ ]:
import gradio as gr
from crewai import Agent, Task, Crew, Process

def run_content_crew(topic: str, audience: str) -> str:
    researcher = Agent(
        role="Researcher",
        goal=f"Research {topic} thoroughly",
        backstory="Expert researcher",
        verbose=False
    )
    writer = Agent(
        role="Writer",
        goal=f"Write engaging content for {audience}",
        backstory="Professional content writer",
        verbose=False
    )
    
    research = Task(
        description=f"Research key facts and insights about: {topic}",
        expected_output="A bulleted research summary with key points",
        agent=researcher
    )
    writing = Task(
        description=f"Write a blog post about {topic} for {audience} audience",
        expected_output="A complete blog post (500 words)",
        agent=writer,
        context=[research]
    )
    
    crew = Crew(
        agents=[researcher, writer],
        tasks=[research, writing],
        process=Process.sequential,
        verbose=False
    )
    
    result = crew.kickoff()
    return str(result)

# demo = gr.Interface(
#     fn=run_content_crew,
#     inputs=[
#         gr.Textbox(label="Topic"),
#         gr.Dropdown(["beginners", "experts", "business executives"], label="Audience")
#     ],
#     outputs=gr.Markdown(label="Generated Content"),
#     title="CrewAI Content Generator"
# )
# demo.launch()


## 21.8 Summary

| Feature | CrewAI API |
|---------|-----------|
| Create agent | `Agent(role, goal, backstory)` |
| Add tools | `tools=[Tool1(), Tool2()]` |
| Define task | `Task(description, expected_output, agent)` |
| Task dependencies | `context=[previous_task]` |
| Sequential crew | `Crew(..., process=Process.sequential)` |
| Hierarchical crew | `Crew(..., process=Process.hierarchical, manager_llm=...)` |
| Run crew | `crew.kickoff()` |

---

**Next:** [Part 22 — LangGraph Deep Dive](Part22_LangGraph_Deep_Dive.ipynb)
